# Креды

In [53]:
# postgres_dev
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
credentials

{'host': '10.6.81.133',
 'port': '5432',
 'user': 'postgres',
 'password': 'tpY7H&sdvsdfsdf7zx9J'}

# Подключение

In [54]:
import geopandas as gpd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
# Подключение к БД
# engine = create_engine(f'postgresql://{credentials.get("user")}:pass@{credentials.get("host")}:{credentials.get("port")}/{credentials.get("password")}')

# Создаем URL через объект (автоматически экранирует)
url = URL.create(
    drivername='postgresql',
    username='postgres',
    password=credentials.get('password'),
    host=credentials.get("host"),
    port=credentials.get("port"),
    database='gisdb_8411_250226',
      query={
        'options': '-c search_path=egip,public'
    }
)

engine = create_engine(url)

# Сохраняем в файл

In [61]:
# sql = """
# select id, layer_id, 
#     ST_Transform(ST_GeomFromWKB(geometry), 4326) as geometry,
#     ST_Transform(ST_GeomFromWKB(centroid), 4326) as centroid
# from public.features_plain
# where layer_alias is NULL
# -- сортируем по геохешу для оптимизации
# order by ST_GeoHash(ST_Transform(ST_GeomFromWKB(centroid), 4326)) 
# limit 1000
# """

sql = """SELECT 
    id, 
    layer_id, 
    geometry,
    centroid
FROM (
    SELECT 
        id, 
        layer_id, 
        ST_Transform(
            ST_SetSRID(ST_GeomFromWKB(geometry), 4326), 
            4326
        ) AS geometry,
        ST_Transform(
            ST_SetSRID(ST_GeomFromWKB(centroid), 4326), 
            4326
        ) AS centroid,
        ST_GeoHash(
            ST_Transform(
                ST_SetSRID(ST_GeomFromWKB(centroid), 4326), 
                4326
            )
        ) AS geohash
    FROM public.features_plain
    WHERE layer_alias IS NULL
) AS subquery

ORDER BY geohash
LIMIT 1000; """

In [62]:
# Читаем в GeoDataFrame, указываем основную геометрию
gdf = gpd.GeoDataFrame.from_postgis(sql, engine, geom_col='geometry', crs='epsg:4326')
# Явно преобразуем колонку centroid в геометрию, если она не была распознана автоматически.
gdf['centroid'] = gpd.GeoSeries.from_wkb(gdf['centroid'])
# Теперь в gdf две колонки с типом 'geometry'
# При сохранении в GeoParquet все геометрические колонки будут записаны.
gdf.to_parquet(r'D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_plain.geoparquet', engine='pyarrow')

# Читаем файл

## список колонок с типом геометрии

In [44]:
# Загружаем ваш GeoParquet файл
gdf = gpd.read_parquet(r'D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_plain.geoparquet')
print(gdf.head())

        id  layer_id                                           geometry  \
0  3496055       165  POLYGON ((37.36723 55.84329, 37.36745 55.8432,...   
1  3496056       165  POLYGON ((37.36723 55.84328, 37.36745 55.84319...   
2  3496057       165  POLYGON ((37.50115 55.86248, 37.50116 55.86248...   
3  3496058       165  POLYGON ((37.50115 55.86248, 37.50116 55.86248...   
4  3496059       165  POLYGON ((37.40096 55.88739, 37.40073 55.88747...   

                    centroid  
0  POINT (37.36687 55.84307)  
1  POINT (37.36687 55.84307)  
2  POINT (37.50151 55.86201)  
3  POINT (37.50151 55.86201)  
4  POINT (37.40083 55.88691)  


In [45]:
# Способ 1: Вывести все колонки, которые являются геометрическими.
# В GeoDataFrame это колонки типа 'geometry'.
geometry_columns = gdf.columns[gdf.dtypes == 'geometry'].tolist()
print("Геометрические колонки:", geometry_columns)

Геометрические колонки: ['geometry', 'centroid']


In [46]:
# Способ 2: Узнать, какая колонка является 'активной' (primary geometry).
# Это основная геометрия, с которой работают по умолчанию.
print("Активная геометрия:", gdf.geometry.name)

Активная геометрия: geometry
